In [ ]:
pip install --upgrade gradio

In [54]:
import gradio as gr
import time

In [55]:
CSS = """
.status-container { display: flex; justify-content: space-around; align-items: center; width: 100%; gap: 10px; }
.status-step { text-align: center; font-weight: bold; color: #888; padding: 10px; border: 3px solid #DDD; border-radius: 15px; flex: 1; transition: all 0.3s ease-in-out; }
.status-step.active { border-color: #3B82F6; color: #3B82F6; transform: scale(1.05); }
.status-step.completed { border-color: #16A34A; color: #16A34A; }
.status-step .icon { font-size: 2em; }
.status-step .text { font-size: 0.9em; }
"""

def create_status_html(icon, text, css_class=""):
    return f"""
    <div class="status-step {css_class}">
        <div class="icon">{icon}</div>
        <div class="text">{text}</div>
    </div>
    """

In [ ]:
def processar_audio(caminho_do_audio):
    if caminho_do_audio is None:
        return; yield

    # ETAPA 1: MIXAGEM
    yield {
        status_upload: gr.update(value=create_status_html("⬆️", "Upload", "active")),
    }
    time.sleep(10)
    # FUNÇÃO DE MIXAGEM

    # ETAPA 2: TRADUÇÃO
    yield {
        status_upload: gr.update(value=create_status_html("⬆️", "Upload", "completed")),
        status_translation: gr.update(value=create_status_html("🌐", "Tradução", "active")),
    }
    time.sleep(10)
    # FUNÇÃO DE TRADUCAO

    # ETAPA 3: PROCESSAMENTO
    yield {
        status_translation: gr.update(value=create_status_html("🌐", "Tradução", "completed")),
        status_processing: gr.update(value=create_status_html("🧠", "Processamento", "active")),
    }
    time.sleep(10)
    # FUNÇÃO DE PROCESSAMENTO

    # ETAPA 4: RESULTADO
    yield {
        status_processing: gr.update(value=create_status_html("🧠", "Processamento", "completed")),
        status_result: gr.update(value=create_status_html("🎶", "Resultado", "active")),
        audio_output: gr.update(value=caminho_do_audio, visible=True),
        process_button: gr.update(visible=False),
        download_audio_button: gr.update(visible=True)
    }

In [70]:
with gr.Blocks(css=CSS) as demo:
    # Stepper de Status
    with gr.Row(elem_classes="status-container"):
        status_upload = gr.HTML(create_status_html("⬆️", "Upload"))
        status_translation = gr.HTML(create_status_html("🌐", "Tradução"))
        status_processing = gr.HTML(create_status_html("🧠", "Processamento"))
        status_result = gr.HTML(create_status_html("🎶", "Resultado"))

    # Títulos
    gr.Markdown("# Tradução Automática com IA")
    gr.Markdown("### Envie o áudio e deixe a IA traduzir a letra automaticamente.")

    # Componentes de I/O
    with gr.Row():
        audio_input = gr.Audio(type="filepath", show_label=False)
        audio_output = gr.Audio(label="Áudio Processado", visible=False, show_label=False)

    # Botões (com visibilidade inicial definida)
    process_button = gr.Button("Iniciar Processamento")
    download_audio_button = gr.Button("Baixar Música Traduzida ⬇️", visible=False)

    # Função para resetar a interface
    def reset_status():
        return {
            status_upload: gr.update(value=create_status_html("⬆️", "Upload")),
            status_translation: gr.update(value=create_status_html("🌐", "Tradução")),
            status_processing: gr.update(value=create_status_html("🧠", "Processamento")),
            status_result: gr.update(value=create_status_html("🎶", "Resultado")),
            audio_output: gr.update(visible=False, value=None),
            process_button: gr.update(visible=True),
            download_audio_button: gr.update(visible=False)
        }

    # Eventos
    process_button.click(
        fn=processar_audio,
        inputs=audio_input,
        outputs=[status_upload, status_translation, status_processing, status_result, audio_output, process_button, download_audio_button]
    )

    audio_input.upload(
        fn=reset_status,
        inputs=None,
        outputs=[status_upload, status_translation, status_processing, status_result, audio_output, process_button, download_audio_button]
    )


demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9d99ddbd1ad6f7b1b0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
